# Satang The Value - Custom Thai TTS Pipeline (Google Colab T4)
Production-grade Thai Custom TTS training targeting ElevenLabs parity on Colab Free T4 (15GB VRAM) with Verbose DEBUG Logging and Fault-tolerant Guards.

## 1. Environment Setup via `uv` (Ultra Fast < 60s)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set Global Debug Log Level
import os
os.environ["LOG_LEVEL"] = "DEBUG"

# Install uv package manager (takes ~1s)
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

# Clone repo or navigate into project directory
!git clone https://github.com/SatangThevalue/satangthevalue-tts-custom.git /content/satangthevalue-tts-custom
%cd /content/satangthevalue-tts-custom

# Install dependencies with uv --system
!uv pip install --system -e .

# Auto-create required Google Drive directory structure
!mkdir -p /content/drive/MyDrive/tts-project/{01_raw,02_processed,03_checkpoints,04_onnx_exports}

# Run Pipeline Sanity Self-Check
!python tests/test_pipeline_logic.py

## 2. Audio Enhancement (Denoise + De-reverb + Bandwidth Extension)

In [ ]:
# Cleans phone recordings and enhances to 24kHz studio grade
!python -m src.audio.enhance \
    --raw-dir "/content/drive/MyDrive/tts-project/01_raw" \
    --out-dir "/content/drive/MyDrive/tts-project/02_processed/enhanced"

## 3. Silero VAD Slicing with Breath Preservation (3-10s)

In [ ]:
!python -m src.audio.slicer \
    --input "/content/drive/MyDrive/tts-project/02_processed/enhanced" \
    --out-dir "/content/drive/MyDrive/tts-project/02_processed/wavs"

## 4. Faster-Whisper ASR & Thai Tonal G2P

In [ ]:
# 4.1 Auto ASR
!python -m src.asr.transcribe \
    --wavs-dir "/content/drive/MyDrive/tts-project/02_processed/wavs" \
    --output-jsonl "/content/drive/MyDrive/tts-project/02_processed/metadata_raw.jsonl"

# 4.2 Thai Tone Tagging [0-4]
!python -m src.g2p.text_norm \
    --input "/content/drive/MyDrive/tts-project/02_processed/metadata_raw.jsonl" \
    --output "/content/drive/MyDrive/tts-project/02_processed/metadata.jsonl"

## 5. Pack Dataset & Cache to Colab Local NVMe

In [ ]:
from src.utils.pack import pack_dataset, unpack_dataset

# Pack on drive to avoid individual small file latency
tar_path = "/content/drive/MyDrive/tts-project/02_processed/dataset.tar"
pack_dataset("/content/drive/MyDrive/tts-project/02_processed", tar_path)

# Extract to high-speed Colab local disk
unpack_dataset(tar_path, "/content/dataset_local")
print("Local dataset cache prepared!")

## 6. LoRA Fine-Tuning (Colab T4 Safe < 13GB VRAM)

In [ ]:
# Train with 8-bit AdamW + Auto-save to Google Drive every 200 steps
!python -m src.training.finetune_lora \
    --config configs/base_config.yaml \
    --lora-config configs/lora_config.yaml

## 7. ONNX Export & INT8 Quantization

In [ ]:
# 7.1 Export PyTorch graph to ONNX
!python -m src.export.export_onnx \
    --checkpoint "/content/drive/MyDrive/tts-project/03_checkpoints/step_2500/adapter_model.pt" \
    --output "/content/drive/MyDrive/tts-project/04_onnx_exports/model.onnx"

# 7.2 Run onnxslim and INT8 Quantization
!python -m src.export.optimize \
    --input "/content/drive/MyDrive/tts-project/04_onnx_exports/model.onnx" \
    --output "/content/drive/MyDrive/tts-project/04_onnx_exports/model_quant_int8.onnx"

## 8. Test Inference with Studio DSP Mastering & Audio Player

In [ ]:
from IPython.display import Audio, display
from src.inference.engine import TTSEngine

onnx_model = "/content/drive/MyDrive/tts-project/04_onnx_exports/model_quant_int8.onnx"
engine = TTSEngine(onnx_model)

test_prompt = "สวัสดีครับ นี่คือระบบเสียงสังเคราะห์ภาษาไทย คุณภาพสูงเทียบเท่าอีเลฟเว่นแล็บส์"
res = engine.synthesize(test_prompt, output_wav_path="demo_speech.wav")

print(f"Phonemes: {res['phonemes']}")
print(f"RTF: {res['rtf']:.3f} | Elapsed: {res['elapsed_sec']:.2f}s")
display(Audio(res['output_path']))